# API do IDS didático: GET, POST e DELETE com Python

**Redes de Computadores II · Prática em sala**

Neste notebook você vai:

1. **Conhecer a estrutura da API**: endereço, recurso, rotas, mensagens e regras.
2. **Acompanhar uma demonstração** de `GET`, `POST` e `DELETE` funcionando.
3. **Descrever o contrato** em Python: método, caminho, corpo e status esperado de cada operação.
4. **Implementar** funções que enviam requisições `GET`, `POST` e `DELETE`.
5. **Comparar** o resultado esperado com o resultado observado.

> [INFO] **A API é compartilhada pela turma.** Todos enxergam os mesmos alertas. Apague **somente** os alertas que você criou e **não apague** os alertas 1 e 2.

**Como usar:** execute as células em ordem (`Shift + Enter`). Complete as linhas marcadas com `# TODO`, substituindo cada `...` pelo código pedido.

## Estrutura da API

### 1. Endereço e caminho de uma requisição

**Endereço base:** `http://179.198.101.48:8080`. Cada rota é somada a ele, por exemplo: `http://179.198.101.48:8080/alertas`.

```text
notebook (cliente, biblioteca requests)
   │  requisição HTTP: método + caminho + cabeçalhos + corpo JSON
   ▼
nginx, porta 8080 ··········· recebe a conexão da internet e repassa
   ▼
gunicorn + Django REST Framework ··· valida a entrada e executa a operação
   ▼
lista de alertas na memória do servidor
   │
   └──> resposta HTTP: status + cabeçalhos + corpo JSON, de volta ao notebook
```

### 2. O recurso: alerta

| Campo | Tipo | Quem define | No `POST` |
|---|---|---|---|
| `id` | número inteiro | a **API**, ao criar | não envie |
| `origem` | texto | o cliente | obrigatório |
| `tipo` | texto | o cliente | obrigatório |
| `descricao` | texto | o cliente | obrigatório |

```json
{"id": 1, "origem": "192.0.2.10", "tipo": "porta_incomum", "descricao": "Conexão observada em porta não esperada"}
```

### 3. Rotas (o contrato)

| Método | Caminho | Corpo enviado | Sucesso | Erros possíveis |
|---|---|---|---|---|
| `GET` | `/alertas` | nenhum | `200` + lista JSON | — |
| `POST` | `/alertas` | JSON com `origem`, `tipo` e `descricao` | `201` + alerta criado, já com `id` | `422` campo obrigatório ausente ou vazio · `400` JSON malformado |
| `GET` | `/alertas/{id}` | nenhum | `200` + alerta JSON | `404` id inexistente |
| `DELETE` | `/alertas/{id}` | nenhum | `204`, **sem corpo** | `404` id inexistente |
| `GET` | `/metricas` | nenhum | `200` + resumo JSON | — |

- `{id}` representa o número do alerta: `/alertas/1`, `/alertas/2`…
- Um método não listado para um caminho existente recebe **`405 Method Not Allowed`** (exemplos: `POST /metricas`, `DELETE /alertas`). O cabeçalho `Allow` da resposta informa os métodos aceitos.

### 4. Exemplos de mensagens

**`GET /alertas`** → `200 OK`
```json
[
  {"id": 1, "origem": "192.0.2.10", "tipo": "porta_incomum", "descricao": "Conexão observada em porta não esperada"},
  {"id": 2, "origem": "198.51.100.7", "tipo": "tentativas_repetidas", "descricao": "Múltiplas tentativas de conexão"}
]
```

**`POST /alertas`** (requisição)
```http
POST /alertas HTTP/1.1
Content-Type: application/json

{"origem": "203.0.113.8", "tipo": "acesso_suspeito", "descricao": "Acesso observado fora do padrão"}
```
→ `201 Created` (o número do `id` depende de quantos alertas a turma já criou)
```json
{"id": 3, "origem": "203.0.113.8", "tipo": "acesso_suspeito", "descricao": "Acesso observado fora do padrão"}
```

**`POST /alertas` sem `tipo`** → `422 Unprocessable Content`
```json
{"tipo": ["Este campo é obrigatório."]}
```

**`GET /alertas/999`** → `404 Not Found`
```json
{"detail": "Alerta 999 não encontrado."}
```

**`DELETE /alertas/3`** → `204 No Content`, com corpo vazio.

**`GET /metricas`** → `200 OK`
```json
{"total_alertas": 2, "por_tipo": {"porta_incomum": 1, "tentativas_repetidas": 1}}
```

### 5. Regras de funcionamento

- **Fala JSON:** as respostas vêm com `Content-Type: application/json`. No `POST`, o corpo deve ser JSON com esse mesmo cabeçalho.
- **O `id` é da API:** ela atribui um número novo a cada criação e **nunca reaproveita** um número apagado.
- **Memória, não banco de dados:** se a API for reiniciada, volta a ter só os alertas 1 e 2.
- **Compartilhada e sem login:** qualquer cliente pode consultar, criar e apagar; todos veem a mesma lista.

### 6. Como consumir com Python (`requests`)

| Para… | Use |
|---|---|
| enviar `GET` | `requests.get(url, timeout=TIMEOUT)` |
| enviar `POST` com JSON | `requests.post(url, json=dados, timeout=TIMEOUT)` |
| enviar `DELETE` | `requests.delete(url, timeout=TIMEOUT)` |
| ler o status | `resposta.status_code` (ex.: `200`) |
| ler o corpo JSON | `resposta.json()` → lista ou dicionário Python |
| ler o corpo como texto | `resposta.text` (vazio no `204`) |

> O parâmetro `json=dados` faz o que, no JavaScript, fazíamos à mão: converte o dicionário em texto JSON (`JSON.stringify`) e adiciona `Content-Type: application/json`.

## Preparação

Execute as três células abaixo. Na primeira, troque o apelido.

> Se aparecer `ModuleNotFoundError: No module named 'requests'`, execute `%pip install requests` numa célula e tente de novo.

In [8]:
import json

import requests

# Endereço da API da turma. Para usar uma API no seu computador: "http://127.0.0.1:8000"
BASE_URL = "http://179.198.101.48:8080"

# Tempo máximo de espera por uma resposta, em segundos: toda requisição de rede precisa de limite.
TIMEOUT = 5

# TODO: troque pelo seu apelido ou iniciais (não use o nome completo: a API é pública).
MEU_APELIDO = "anônimo"

In [9]:
# Funções auxiliares: não precisa alterar esta célula, apenas execute.

def mostrar(resposta):
    """Mostra a requisição enviada e a resposta recebida, como a aba Network do DevTools."""
    if not isinstance(resposta, requests.Response):
        print("[WARNING] Ainda não há resposta HTTP aqui. Complete o TODO da função usada.")
        return
    print(f"{resposta.request.method} {resposta.request.url}")
    print(f"→ {resposta.status_code} {resposta.reason}")
    if resposta.status_code == 405:
        print(f"Métodos aceitos neste caminho (Allow): {resposta.headers.get('Allow')}")
    if not resposta.text:
        print("Corpo: (vazio)")
        return
    try:
        print("Corpo JSON:\n" + json.dumps(resposta.json(), indent=2, ensure_ascii=False))
    except ValueError:
        print("Corpo:", resposta.text[:300])


def confere_status(resposta, esperado):
    """Compara o status recebido com o previsto no contrato. Devolve True se forem iguais."""
    if not isinstance(resposta, requests.Response):
        return False  # mostrar() já avisou que falta completar o TODO
    if resposta.status_code == esperado:
        print(f"[OK] Status {esperado}, como o contrato prevê.")
        return True
    print(f"[ERROR] Esperado {esperado}, recebido {resposta.status_code}. Revise método, caminho e corpo.")
    return False


meu_id = None  # id do alerta que você vai criar na Parte 3

**Teste de conexão.** Se aparecer `200 OK`, a API está acessível.

Se aparecer `ConnectionError` ou `Timeout`, **não houve resposta HTTP**: confira o endereço e a rede. Compare com um `404`, que é uma resposta enviada pelo servidor.

In [10]:
resposta = requests.get(f"{BASE_URL}/metricas", timeout=TIMEOUT)
mostrar(resposta)

GET http://179.198.101.48:8080/metricas
→ 200 OK
Corpo JSON:
{
  "total_alertas": 2,
  "por_tipo": {
    "porta_incomum": 1,
    "acesso_suspeito": 1
  }
}


## Demonstração: GET, POST e DELETE

Antes dos exercícios, veja uma requisição de cada tipo funcionando. As células usam o `requests` diretamente, sem funções auxiliares, para que cada parte da requisição e da resposta fique visível:

1. **GET** consulta a lista de alertas.
2. **POST** cria um alerta de demonstração.
3. **DELETE** apaga esse mesmo alerta e confirma a exclusão.

Execute em ordem: o DELETE usa o `id` que a API devolveu no POST.

### Demonstração 1 · `GET /alertas`

O cliente pede a coleção de alertas. A API responde `200 OK` e uma **lista JSON**, que `resposta.json()` transforma numa lista de dicionários Python.

In [17]:
resposta = requests.get(f"{BASE_URL}/alertas", timeout=TIMEOUT)

print("Requisição:", resposta.request.method, resposta.request.url)
print("Status:", resposta.status_code, resposta.reason)  # 200 OK
print("Content-Type:", resposta.headers["Content-Type"])  # application/json

alertas = resposta.json()  # texto JSON -> lista de dicionários Python
print("Quantidade de alertas:", len(alertas))
if alertas:
    print("Primeiro alerta:", alertas[0])
    print("Tipo do primeiro alerta:", alertas[0]["tipo"])

Requisição: GET http://179.198.101.48:8080/alertas
Status: 200 OK
Content-Type: application/json
Quantidade de alertas: 4
Primeiro alerta: {'id': 1, 'origem': '192.0.2.10', 'tipo': 'porta_incomum', 'descricao': 'Conexão observada em porta não esperada'}
Tipo do primeiro alerta: porta_incomum


### Demonstração 2 · `POST /alertas`

O cliente monta um dicionário com `origem`, `tipo` e `descricao`. O parâmetro `json=` converte o dicionário em texto JSON e adiciona o cabeçalho `Content-Type: application/json`. A API responde `201 Created` com o alerta criado, **já com o `id`** que ela escolheu.

In [14]:
novo_alerta = {
    "origem": "203.0.113.8 - iGOR",
    "tipo": "acesso_suspeito",
    "descricao": f"Demonstração · {MEU_APELIDO}",
}
resposta = requests.post(f"{BASE_URL}/alertas", json=novo_alerta, timeout=TIMEOUT)

print("Requisição:", resposta.request.method, resposta.request.url)
print("Content-Type enviado:", resposta.request.headers["Content-Type"])
# No texto JSON enviado, letras acentuadas aparecem como ç, ã...: é a forma ASCII de escrevê-las.
print("Corpo enviado:", resposta.request.body.decode())
print("Status:", resposta.status_code, resposta.reason)  # 201 Created

id_demo = None
if resposta.status_code == 201:
    alerta_criado = resposta.json()
    id_demo = alerta_criado["id"]  # quem escolhe o id é a API
    print("Alerta criado:", alerta_criado)

Requisição: POST http://179.198.101.48:8080/alertas
Content-Type enviado: application/json
Corpo enviado: {"origem": "203.0.113.8 - iGOR", "tipo": "acesso_suspeito", "descricao": "Demonstra\u00e7\u00e3o \u00b7 an\u00f4nimo"}
Status: 201 Created
Alerta criado: {'id': 7, 'origem': '203.0.113.8 - iGOR', 'tipo': 'acesso_suspeito', 'descricao': 'Demonstração · anônimo'}


### Demonstração 3 · `DELETE /alertas/{id}`

O cliente apaga o alerta criado na demonstração 2. A API responde `204 No Content`: deu certo e **não há corpo**. Um `GET` logo depois confirma a exclusão com `404 Not Found`.

In [15]:
if id_demo is None:
    print("[WARNING] Execute a demonstração 2 (POST) primeiro.")
else:
    resposta = requests.delete(f"{BASE_URL}/alertas/{id_demo}", timeout=TIMEOUT)
    print("Requisição:", resposta.request.method, resposta.request.url)
    print("Status:", resposta.status_code, resposta.reason)  # 204 No Content
    print("Corpo recebido:", repr(resposta.text))  # '' -> o 204 não tem corpo

    confirmacao = requests.get(f"{BASE_URL}/alertas/{id_demo}", timeout=TIMEOUT)
    print("\nGET depois do DELETE:", confirmacao.status_code, confirmacao.reason)  # 404 Not Found
    print("Corpo:", confirmacao.json())

Requisição: DELETE http://179.198.101.48:8080/alertas/7
Status: 204 No Content
Corpo recebido: ''

GET depois do DELETE: 404 Not Found
Corpo: {'detail': 'Alerta 7 não encontrado.'}


## Parte 1 · Descreva o contrato

Usando a **Estrutura da API** acima, preencha cada linha da lista `CONTRATO`:

| Campo | O que colocar |
|---|---|
| método | `"GET"`, `"POST"` ou `"DELETE"` |
| caminho | começa com `/`, por exemplo `"/alertas"` ou `"/alertas/999"` |
| corpo | `None` (sem corpo), `CORPO_VALIDO` ou `CORPO_SEM_TIPO` |
| status esperado | número inteiro: `200`, `201`, `204`, `404`, `405` ou `422` |

Para o **alerta que você vai criar**, escreva `{id}` no caminho (exemplo: `"/alertas/{id}"`). Na Parte 5, o `{id}` é trocado pelo número que a API devolver.

A primeira linha já está resolvida. Substitua os `...` das demais.

### Códigos de status desta API

O **primeiro dígito** indica a família da resposta:
`2xx` = deu certo · `4xx` = problema no pedido do cliente · `5xx` = falha no servidor.

| Código | Nome | Significado | Quando aparece nesta API |
|---|---|---|---|
| `200` | OK | A consulta deu certo e o corpo traz os dados | `GET /alertas`, `GET /alertas/{id}`, `GET /metricas` |
| `201` | Created | Um recurso novo foi criado; o corpo traz o recurso com o `id` | `POST /alertas` com os três campos |
| `204` | No Content | Deu certo e **não há corpo** na resposta | `DELETE /alertas/{id}` de um alerta existente |
| `400` | Bad Request | O corpo enviado não é um JSON válido (erro de sintaxe) | `POST /alertas` com JSON malformado, ex.: `{tipo: "x"}` |
| `404` | Not Found | O recurso pedido não existe | `GET` ou `DELETE` em `/alertas/999`, ou num alerta já apagado |
| `405` | Method Not Allowed | O caminho existe, mas não aceita esse método; o cabeçalho `Allow` lista os aceitos | `POST /metricas`, `DELETE /alertas` |
| `422` | Unprocessable Content | O JSON é válido, mas não cumpre o contrato (campo obrigatório ausente ou vazio) | `POST /alertas` sem `origem`, `tipo` ou `descricao` |
| `5xx` | Server Error | Falha interna do servidor; não é culpa do pedido | Não deveria ocorrer: avise o professor |

**Diferenças que mais confundem:**

- **`404` × `405`:** no `404` o recurso não existe; no `405` o caminho existe, mas o método é que não é permitido.
- **`400` × `422`:** no `400` o texto nem é JSON válido (sintaxe); no `422` o JSON é válido, mas falta um campo exigido (contrato).
- **`200` × `201` × `204`:** os três são sucesso. `201` indica que algo foi criado; `204` indica que não há corpo, então não use `resposta.json()`.
- **`404` × sem código:** um `404` é uma **resposta** do servidor. Se aparecer `ConnectionError` ou `Timeout`, **nenhuma** resposta HTTP chegou: o problema está no endereço, na rede ou no servidor fora do ar.

> No Python, o código fica em `resposta.status_code` e o nome em `resposta.reason`. No navegador, veja a coluna **Status** da aba Network (F12).

In [ ]:
CORPO_VALIDO = {
    "origem": "203.0.113.8",
    "tipo": "acesso_suspeito",
    "descricao": f"Teste do contrato · {MEU_APELIDO}",
}
CORPO_SEM_TIPO = {
    "origem": "203.0.113.8",
    "descricao": f"Teste sem tipo · {MEU_APELIDO}",
}

CONTRATO = [
    # (o que fazer,                                 método, caminho,    corpo, status esperado)
    ("listar todos os alertas",                      "GET",  "/alertas", None,  200),  # exemplo resolvido
    ("consultar o resumo (métricas)",                ...,    ...,        ...,   ...),
    ("consultar um alerta que não existe (id 999)",  ...,    ...,        ...,   ...),
    ("criar um alerta com os três campos",           ...,    ...,        ...,   ...),
    ("consultar o alerta que você acabou de criar",  ...,    ...,        ...,   ...),
    ("criar um alerta sem o campo tipo",             ...,    ...,        ...,   ...),
    ("apagar o alerta que você criou",               ...,    ...,        ...,   ...),
    ("apagar de novo o mesmo alerta",                ...,    ...,        ...,   ...),
    ("apagar a coleção inteira de alertas",          ...,    ...,        ...,   ...),
]
print(f"{len(CONTRATO)} operações descritas.")

## Parte 2 · GET: consultar dados

**Exemplo resolvido:** a função abaixo lista todos os alertas. Use-a como modelo nos exercícios.

In [ ]:
def listar_alertas():
    return requests.get(f"{BASE_URL}/alertas", timeout=TIMEOUT)


resposta = listar_alertas()
mostrar(resposta)
confere_status(resposta, 200)

**Exercício 2.1.** `resposta.json()` transforma o corpo JSON numa **lista de dicionários** Python. Percorra a lista e imprima o `id` e o `tipo` de cada alerta.

In [ ]:
alertas = listar_alertas().json()
print(f"A API devolveu {len(alertas)} alertas.")

for alerta in alertas:
    # TODO: imprima o id e o tipo deste alerta, por exemplo: 1 porta_incomum
    ...

**Exercício 2.2.** Complete as duas funções seguindo o modelo de `listar_alertas()`. Não esqueça o `timeout=TIMEOUT`.

In [ ]:
def buscar_alerta(alerta_id):
    # TODO: envie um GET para /alertas/<alerta_id> e devolva a resposta
    return ...


def consultar_metricas():
    # TODO: envie um GET para /metricas e devolva a resposta
    return ...

In [ ]:
print("— Alerta 1 —")
resposta = buscar_alerta(1)
mostrar(resposta)
confere_status(resposta, 200)  # se der 404, alguém apagou o alerta 1: avise o professor

print("\n— Alerta 999 —")
resposta = buscar_alerta(999)
mostrar(resposta)
confere_status(resposta, 404)

print("\n— Métricas —")
resposta = consultar_metricas()
mostrar(resposta)
confere_status(resposta, 200)

## Parte 3 · POST: criar um alerta

**Exercício 3.1.** Complete `criar_alerta()`. O dicionário já está montado: falta enviá-lo no corpo JSON com `requests.post(url, json=..., timeout=TIMEOUT)`.

Lembre: **não envie o `id`**. Quem atribui é a API, e você o lê na resposta.

In [ ]:
def criar_alerta(origem, tipo, descricao):
    novo_alerta = {
        "origem": origem,
        "tipo": tipo,
        "descricao": descricao,
    }
    # TODO: envie um POST para /alertas com novo_alerta no corpo JSON e devolva a resposta
    return ...

In [ ]:
resposta = criar_alerta("203.0.113.8", "acesso_suspeito", f"Alerta de teste · {MEU_APELIDO}")
mostrar(resposta)

if confere_status(resposta, 201):
    meu_id = resposta.json()["id"]  # o id vem da resposta: nunca adivinhe
    print(f"\nSeu alerta recebeu o id {meu_id}.")

In [ ]:
# Um GET posterior confirma que o alerta foi mesmo criado.
if meu_id is None:
    print("[WARNING] Crie seu alerta na célula anterior primeiro.")
else:
    resposta = buscar_alerta(meu_id)
    mostrar(resposta)
    confere_status(resposta, 200)

**Exercício 3.2.** Envie um `POST` **sem** o campo `tipo`, usando `requests.post` diretamente. Qual status o contrato prevê? O alerta é criado?

In [ ]:
sem_tipo = {"origem": "203.0.113.8", "descricao": f"Sem tipo · {MEU_APELIDO}"}

# TODO: envie um POST para /alertas com sem_tipo no corpo JSON
resposta = ...

mostrar(resposta)
confere_status(resposta, 422)

## Parte 4 · DELETE: apagar o seu alerta

**Exercício 4.1.** Complete `apagar_alerta()` com `requests.delete(url, timeout=TIMEOUT)`.

- Sucesso: `204 No Content`. Deu certo e **não há corpo**, então não chame `resposta.json()`.
- Alerta inexistente: `404 Not Found`.

> Apague **somente** o alerta que você criou (`meu_id`).

In [ ]:
def apagar_alerta(alerta_id):
    # TODO: envie um DELETE para /alertas/<alerta_id> e devolva a resposta
    return ...

In [ ]:
if meu_id is None:
    print("[WARNING] Crie seu alerta na Parte 3 primeiro.")
else:
    resposta = apagar_alerta(meu_id)
    mostrar(resposta)
    confere_status(resposta, 204)

**Depois do DELETE:** consulte e tente apagar de novo o mesmo alerta. O que muda nas respostas?

In [ ]:
if meu_id is None:
    print("[WARNING] Crie seu alerta na Parte 3 primeiro.")
else:
    print("— GET depois do DELETE —")
    resposta = buscar_alerta(meu_id)
    mostrar(resposta)
    confere_status(resposta, 404)

    print("\n— DELETE de novo —")
    resposta = apagar_alerta(meu_id)
    mostrar(resposta)
    confere_status(resposta, 404)

## Parte 5 · Matriz de testes: esperado × observado

A função abaixo percorre o seu `CONTRATO` (Parte 1), envia cada requisição e compara o status **esperado** (o que você descreveu) com o **observado** (o que a API respondeu).

- [OK] a sua descrição confere com a API;
- [ERROR] há divergência: descubra se o erro está no método, no caminho, no corpo ou no status e corrija a Parte 1;
- [PENDING] a linha ainda tem `...`.

A linha de criação gera um alerta de teste, e a linha de exclusão o apaga. Se alterar a Parte 1, execute a célula dela de novo antes desta.

In [18]:
# Não precisa alterar esta célula, apenas execute.

def executar_contrato(contrato):
    """Envia cada requisição descrita e compara o status esperado com o observado."""
    id_criado = None
    conferem = 0
    for o_que, metodo, caminho, corpo, esperado in contrato:
        if ... in (metodo, caminho, corpo, esperado):
            print(f"[PENDING] {o_que}: linha ainda não preenchida")
            continue
        if not str(caminho).startswith("/"):
            print(f"[ERROR] {o_que}: o caminho deve começar com /")
            continue
        if "{id}" in caminho:
            if id_criado is None:
                print(f"[WARNING] {o_que}: nenhum alerta foi criado antes desta linha")
                continue
            caminho = caminho.replace("{id}", str(id_criado))

        resposta = requests.request(metodo, BASE_URL + caminho, json=corpo, timeout=TIMEOUT)
        if resposta.status_code == 201:
            id_criado = resposta.json()["id"]

        if resposta.status_code == esperado:
            conferem += 1
            marca = "[OK]"
        else:
            marca = "[ERROR]"
        print(f"{marca:7} {metodo:6} {caminho:15} esperado {esperado} · observado {resposta.status_code}  ({o_que})")

    print(f"\n{conferem} de {len(contrato)} linhas conferem com a API.")


executar_contrato(CONTRATO)

[OK]    GET    /alertas        esperado 200 · observado 200  (listar todos os alertas)
[PENDING] consultar o resumo (métricas): linha ainda não preenchida
[PENDING] consultar um alerta que não existe (id 999): linha ainda não preenchida
[PENDING] criar um alerta com os três campos: linha ainda não preenchida
[PENDING] consultar o alerta que você acabou de criar: linha ainda não preenchida
[PENDING] criar um alerta sem o campo tipo: linha ainda não preenchida
[PENDING] enviar POST para o resumo (métricas): linha ainda não preenchida
[PENDING] apagar o alerta que você criou: linha ainda não preenchida
[PENDING] apagar de novo o mesmo alerta: linha ainda não preenchida
[PENDING] apagar a coleção inteira de alertas: linha ainda não preenchida

1 de 10 linhas conferem com a API.


## Para responder

Escreva suas respostas na célula abaixo (dê dois cliques para editar).

1. Um `404` e um `ConnectionError` significam a mesma coisa? O que cada um diz sobre a comunicação com o servidor?
2. Por que o `id` do seu alerta provavelmente **não** foi 3?
3. Por que `resposta.json()` falharia na resposta de um `DELETE` bem-sucedido?
4. Você apagou o mesmo alerta duas vezes e recebeu `204` e depois `404`. O estado da API mudou na segunda vez?
5. Nas suas observações, o que indicam `404`, `405` e `422`?
6. Se o professor reiniciar a API, o que acontece com os alertas criados pela turma? Por quê?

**Suas respostas:**

1.
2.
3.
4.
5.
6.